# Agentic RAG with LangGraph

This notebook demonstrates building an Agentic RAG system using LangGraph with autonomous agents.

## Setup

In [ ]:
!pip install langgraph langchain langchain-openai langchain-community

## Import Libraries

In [ ]:
import os
from langgraph.graph import StateGraph, END
from langchain_community.tools import Tool
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.chat_models import ChatOpenAI
from pydantic import BaseModel
from typing import List, TypedDict, Optional
import json

os.environ["OPENAI_API_KEY"] = "your-api-key-here"

## Define Agent State

In [ ]:
class AgentState(TypedDict):
    """State for the Agentic RAG agent."""
    question: str
    plan: List[str]
    retrieved_docs: List
    evaluation: str
    final_answer: Optional[str]
    iterations: int

## Create Tools

In [ ]:
# Initialize components
llm = ChatOpenAI(model="gpt-4", temperature=0)
embeddings = OpenAIEmbeddings()

# Create vector store (using existing documents)
vectorstore = Chroma.from_documents(
    documents=documents,  # From previous notebook
    embedding=embeddings
)

def vector_search(query: str) -> str:
    """Search the knowledge base."""
    docs = vectorstore.similarity_search(query, k=4)
    return "\n\n".join([f"Doc {i+1}: {doc.page_content[:200]}" for i, doc in enumerate(docs)])

def analyze_context(query: str, context: str) -> str:
    """Analyze if context answers the question."""
    prompt = f"Question: {query}\n\nContext:\n{context}\n\nDoes this answer the question?"
    return llm.predict(prompt)

tools = [
    Tool(
        name="vector_search",
        func=vector_search,
        description="Search internal knowledge base"
    ),
    Tool(
        name="analyze_context",
        func=analyze_context,
        description="Analyze if retrieved context answers the question"
    )
]

## Define Agent Nodes

In [ ]:
def planner_node(state: AgentState) -> AgentState:
    """Plan retrieval strategy."""
    prompt = f"""Given this question: {state['question']}\n\nCreate a plan to answer it.\nRespond as a JSON list of steps."""
    response = llm.predict(prompt)
    try:
        plan = json.loads(response)
    except:
        plan = ["Search knowledge base", "Analyze results", "Generate answer"]
    return {"plan": plan, "iterations": state.get("iterations", 0)}

def retriever_node(state: AgentState) -> AgentState:
    """Execute retrieval."""
    docs = vectorstore.similarity_search(state['question'], k=4)
    return {"retrieved_docs": docs}

def evaluator_node(state: AgentState) -> AgentState:
    """Evaluate retrieval quality."""
    context = "\n\n".join([doc.page_content for doc in state['retrieved_docs']])
    prompt = f"""Question: {state['question']}\n\nContext:\n{context}\n\nIs this sufficient?\n\nRespond: 'sufficient', 'need_more', or 'refine'"""
    evaluation = llm.predict(prompt).strip().lower()
    return {"evaluation": evaluation}

def generator_node(state: AgentState) -> AgentState:
    """Generate final answer."""
    context = "\n\n".join([doc.page_content for doc in state['retrieved_docs']])
    prompt = f"""Based on the context, answer the question.\n\nContext:\n{context}\n\nQuestion: {state['question']}\n\nAnswer:"""
    answer = llm.predict(prompt)
    return {"final_answer": answer}

## Build the Graph

In [ ]:
workflow = StateGraph(AgentState)

workflow.add_node("planner", planner_node)
workflow.add_node("retrieve", retriever_node)
workflow.add_node("evaluate", evaluator_node)
workflow.add_node("generate", generator_node)

workflow.set_entry_point("planner")
workflow.add_edge("planner", "retrieve")
workflow.add_edge("retrieve", "evaluate")

def should_continue(state: AgentState) -> str:
    """Decide whether to continue or finish."""
    if state.get("evaluation", "") == "need_more":
        if state.get("iterations", 0) < 2:
            return "retry"
    return "generate"

workflow.add_conditional_edges(
    "evaluate",
    should_continue,
    {"retry": "planner", "generate": "generate"}
)

workflow.add_edge("generate", END)

app = workflow.compile()

## Run Agentic RAG

In [ ]:
result = app.invoke({
    "question": "What are the key benefits of RAG over pure LLMs?",
    "plan": [],
    "retrieved_docs": [],
    "evaluation": "",
    "final_answer": None,
    "iterations": 0
})

print("Question:", result["question"])
print("\nPlan:", result.get("plan", []))
print("\nIterations:", result.get("iterations", 0))
print("\nAnswer:", result["final_answer"])

## Visualize the Graph

In [ ]:
# Display the graph structure
print(app.get_graph().draw_ascii())

## Try Different Queries

In [ ]:
queries = [
    "How does RAG reduce hallucinations?",
    "What is the difference between classic RAG and agentic RAG?"
]

for q in queries:
    result = app.invoke({
        "question": q,
        "plan": [],
        "retrieved_docs": [],
        "evaluation": "",
        "final_answer": None,
        "iterations": 0
    })
    print(f"Q: {q}")
    print(f"A: {result['final_answer']}\n")

## Next Steps

- Add more tools (web search, calculator)
- Implement multi-agent collaboration
- Add memory for conversation history
- Add error handling and retry logic